# Noise generation

Load a trained drone-noise generator from the zoo, drive it with a synthetic RPS trajectory, and compare the render against a real recording. For interactive sliders (embedding walk, jitter, wind channel), use `generator_lab.ipynb` — this notebook is the thin scripted path over the same models.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import tdseries as td

from plots import dwym, explore

In [ ]:
import os

import data_processing.streams  # noqa: F401 — loads the .env credentials

if not os.environ.get("AWS_ACCESS_KEY_ID"):
    raise RuntimeError(
        "No R2 credentials. The cells below stream dload datasets. "
        "Fill .env at the repo root (see docs/data-and-artifacts.md), then rerun."
    )

## 1 · Noise-gen checkpoints in the zoo

`zoo.checkpoints(task="noise_generation")` lists the trained generator experiments. `zoo.load` rebuilds the model from its experiment config and pulls the checkpoint.

In [ ]:
import zoo

[row["experiment"] for row in zoo.checkpoints(task="noise_generation")]

In [ ]:
gen = zoo.load("e6_noisegen_baseline")
gen

## 2 · A real cruise slice

The slice supplies the comparison audio and the array geometry — published DREGON frames carry `mic_pos` / `rotor_pos` entries.

In [ ]:
rec = explore.pick("DREGON-frames", "free-flight_nosource_room1")
clip = rec.time[20.0:22.0].shift(-20.0)  # 2 s of stable flight, re-based to t=0

## 3 · A synthetic RPS excitation

`rps_synthesis.generate_intermittent` makes a realistic "steady, then a short maneuver" trajectory. The 16 kHz rate matches the generator's per-sample RPS input.

In [ ]:
from data_processing import rps_synthesis

rps = rps_synthesis.generate_intermittent(2.0, 16000, drone_profile=0.0, rng=0)
dwym(td.Frame({"rps": td.uniform(rps, 16000, dims=("rotor", "time"))}))

## 4 · Render

The codec inside the `FrameModel` turns `rps` + geometry into the model call; `meta.drone` selects the per-drone conditioning code.

In [ ]:
inp = td.Frame({
    "rps": td.uniform(rps.astype(np.float32), 16000, dims=("rotor", "time")),
    "mic_pos": clip["mic_pos"], "rotor_pos": clip["rotor_pos"],
    "meta": td.Frame({"drone": "dregon"}),
})
generated = gen(inp)

## 5 · Real vs generated

A dict of two bare-audio frames routes to the real-vs-generated spectrogram grid, with one audio player per row.

In [ ]:
from data_processing.frames import resample_audio_series

real16 = td.Frame({"audio": resample_audio_series(clip["audio"], 16000)})
dwym({"real": real16, "generated": td.Frame({"audio": generated["audio"]})}, fmax=4000)

## 6 · Drive it with the real RPS

Same generator, excited by the recording's own telemetry: the harmonic tracks now line up with the real spectrogram.

In [ ]:
times = np.arange(int(2.0 * 16000)) / 16000
real_rps = np.asarray(clip["rps"].interpolate(times), dtype=np.float32)
gen2 = gen(inp.with_entry("rps", td.uniform(real_rps, 16000, dims=("rotor", "time"))))
dwym({"real": real16, "generated": td.Frame({"audio": gen2["audio"]})}, fmax=4000)

The E6 sibling arms (`e6_noisegen_randphase`, `e6_noisegen_jitter*`, the per-drone variants) load the same way — swap the experiment name in step 1. See `docs/notebook-primitives-tutorial.md` for the primitives, and `generator_lab.ipynb` for the interactive lab.